# QReCC Retrieval Index Creator

This notebook prepares the sparse and dense retrieval indexes for QReCC.

### Local checkout setup

Use this public checkout on a local machine or a prepared Colab runtime. The next cell locates it; it does not clone a private repository, mount Drive, or request credentials.

In [ ]:
from pathlib import Path
import sys
import os

# Use an existing public checkout, locally or in Colab.
# No private repository, deploy key, or Drive mount is required.
ROOT = Path(os.environ.get("ROCC_ROOT", os.environ.get("MASTER_THESIS_PROJECT_ROOT", Path.cwd()))).resolve()
if not (ROOT / "experiments" / "rocc").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "experiments" / "rocc").exists())
os.environ["ROCC_ROOT"] = str(ROOT)
os.environ["MASTER_THESIS_PROJECT_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT / "experiments"))
print("project:", ROOT)


### Runtime setup

Locates the `experiments` directory, exposes the ROCC API, and initializes the local or Colab runtime.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

EXPERIMENTS_DIR = ROOT / "experiments"
if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

import rocc
from rocc.paths import project_path

notebook_runtime = rocc.bootstrap_notebook()
colab_runtime = notebook_runtime.qrecc_runtime
pd.set_option("display.max_colwidth", 180)
display(pd.DataFrame([notebook_runtime.to_dict()]))


### Prepare public QReCC inputs

Reuse local processed data and complete downloaded indices. If numeric-ID train/test splits are missing, download and prepare the public passage corpus; index files alone do not provide the gold-ID mapping.

In [ ]:
QRECC_DATA_DIR = project_path("experiments", "data", "qrecc")
qrecc_resources = rocc.resolve_qrecc_resources(QRECC_DATA_DIR)
qrecc_artifacts_before = rocc.inspect_qrecc_artifacts(qrecc_resources)
qrecc_processed_splits_ready = all(p.exists() for p in (
    qrecc_resources.processed_train_json, qrecc_resources.processed_test_json,
    qrecc_resources.collection_manifest_json))
qrecc_downstream_ready = (qrecc_artifacts_before.processed_collection_ready
    or qrecc_artifacts_before.bm25_index_ready or qrecc_artifacts_before.ance_index_ready)
# Numeric QReCC gold IDs require processed splits even with a prebuilt index.
need_passages = not qrecc_downstream_ready or not qrecc_processed_splits_ready
qrecc_resources = rocc.ensure_qrecc_resources(
    data_dir=QRECC_DATA_DIR, progress=True, download_passages=need_passages,
    extract_passages=need_passages, timeout=1000)
qrecc_artifacts = rocc.inspect_qrecc_artifacts(qrecc_resources)
display(pd.DataFrame([qrecc_artifacts.to_dict()]))


### Prepare numeric-ID collection and splits

Build these from the public QReCC files, or reuse prepared splits with a complete local index. No Azure access is needed. A fresh collection is a large disk operation.

In [ ]:
qrecc_artifacts_before_build = rocc.inspect_qrecc_artifacts(qrecc_resources)
if qrecc_artifacts_before_build.processed_collection_ready:
    qrecc_collection = rocc.load_qrecc_collection_build_result(qrecc_resources)
elif qrecc_resources.passages_dir.exists():
    qrecc_collection = rocc.build_qrecc_collection_and_splits(qrecc_resources, progress=True)
else:
    qrecc_collection = rocc.load_qrecc_collection_build_result(qrecc_resources, require_collection=False)
if not qrecc_collection.collection_jsonl.exists():
    print("Using prepared splits and local index; full collection is not required.")
display(pd.DataFrame([qrecc_collection.to_dict()]))


### Gold-evaluable QReCC splits

Filters the processed train and test splits to turns with at least one gold passage ID. The resulting files contain only retrieval-evaluable queries.

In [6]:
import json
from pathlib import Path
from tqdm.auto import tqdm

QRECC_COLLECTION_DIR = qrecc_collection.collection_jsonl.parent
QRECC_GOLD_TRAIN_JSON = qrecc_collection.processed_train_json.with_name("qrecc-training.gold.json")
QRECC_GOLD_TEST_JSON = qrecc_collection.processed_test_json.with_name("qrecc-test.gold.json")


def positive_ctx_ids(row: dict) -> list[str]:
    ids = row.get("positive_ctx_passage_ids")
    if ids is None:
        raise KeyError("positive_ctx_passage_ids is missing. Run build_qrecc_collection_and_splits first.")
    return ids


def write_gold_only_processed_split(input_path: Path, output_path: Path, split: str) -> dict[str, object]:
    rows = json.loads(input_path.read_text(encoding="utf-8"))
    filtered = [
        row
        for row in tqdm(rows, desc=f"filter {split}", unit="turn")
        if positive_ctx_ids(row)
    ]
    output_path.write_text(
        json.dumps(filtered, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return {
        "split": split,
        "input": str(input_path),
        "output": str(output_path),
        "total": len(rows),
        "kept": len(filtered),
        "dropped": len(rows) - len(filtered),
        "drop_rate": round((len(rows) - len(filtered)) / len(rows), 4) if rows else 0.0,
    }


qrecc_gold_filter_stats = [
    write_gold_only_processed_split(
        qrecc_collection.processed_train_json,
        QRECC_GOLD_TRAIN_JSON,
        "train",
    ),
    write_gold_only_processed_split(
        qrecc_collection.processed_test_json,
        QRECC_GOLD_TEST_JSON,
        "test",
    ),
]

display(pd.DataFrame(qrecc_gold_filter_stats))


,split,input,output,total,kept,dropped,drop_rate
0,train,/experiments/data/qrecc/collection/qrecc-training.json,/experiments/data/qrecc/collection/qrecc-training.gold.json,63501,29596,33905,0.5339
1,test,/experiments/data/qrecc/collection/qrecc-test.json,/experiments/data/qrecc/collection/qrecc-test.gold.json,16451,8209,8242,0.5010


The gold filter retains **29,596 of 63,501 training turns** and **8,209 of 16,451 test turns**. The remaining 53.39% and 50.10% lack gold passage IDs and are therefore excluded from retrieval evaluation. The thesis uses the **8,209 test queries** for transfer evaluation and does not use QReCC training queries to train or select ROCC.

### QReCC source examples

Loads the processed gold-evaluable QReCC split and selects one representative turn from each source dataset for inspection.

In [7]:
qrecc_frame = rocc.load_qrecc_frame(
    qrecc_resources,
    prefer_processed=True,
    gold_only=True,
)

qrecc_source_samples = rocc.sample_qrecc_turns_by_source(
    qrecc_frame,
    per_source=1,
    #seed=455,
)

display(qrecc_source_samples)

,conversation_source,split,conv_id,turn_id,question,rewrite,answers,positive_ctx_passage_ids
0,nq,train,9817,5,and why,why was the national park conservation association established,The mission of the National Parks Conservation Association is to protect and enhance America's National Parks for present and future generations.,[54538114]
1,quac,train,6442,5,What languages and cultures did he helped with?,What languages and cultures did Benjamin Lee Whorf help with?,"At first Benjamin Lee Whorf was drawn to the study of Biblical Hebrew, but he quickly went on to study the indigenous languages of Mesoamerica on his own.",[54381058]
2,trec,test,75,3,What was their importance in native cultures,What was the importance of turkey in native cultures?,ative Americans have hunted wild turkey for its meat as early as 1000 A.D. They made turkey whistles out of turkey wing bones.,[54541736]


The examples confirm consistent processing across the NQ, QuAC, and TREC source datasets. Each row preserves the conversational question, standalone rewrite, reference answer, and gold passage ID required for retrieval evaluation.

### Gold passage verification

Loads the local QReCC passage store when available and resolves each sampled gold passage ID to its text. Passage inspection is skipped when the large collection was intentionally not restored.

In [ ]:
if qrecc_collection.collection_jsonl.exists():
    qrecc_passage_store = rocc.load_qrecc_passage_store(
        qrecc_collection.collection_jsonl,
        progress=True,
    )
else:
    qrecc_passage_store = None
    print("Passage Store Lookup skipped: qrecc_collection.jsonl is not local.")

if qrecc_passage_store is not None:
    qrecc_sample_passages = rocc.lookup_qrecc_passages_for_turns(
        qrecc_source_samples,
        qrecc_passage_store,
    )

    display(pd.DataFrame([qrecc_passage_store.to_dict()]))
    display(qrecc_sample_passages)
else:
    qrecc_sample_passages = None
    print("Passage examples skipped: local corpus was intentionally not restored.")


,collection_jsonl,offsets_path,num_passages,offsets_ram_mib
0,/experiments/data/qrecc/collection/qrecc_collection.jsonl,/experiments/data/qrecc/collection/qrecc_collection.offsets.u64,54573064,416.36


,conversation_source,split,conv_id,turn_id,question,rewrite,positive_ctx_rank,passage_id,row_index,id,raw_passage_id,contents
0,nq,train,9817,5,and why,why was the national park conservation association established,1,54538114,54538114,54538114,http://web.archive.org/web/20191204220627id_/https://www.charitynavigator.org/index.cfm?bay=search.summary&orgid=4192_p2,"97.00 xml version=""1.0"" encoding=""utf-8""? svg PUBLIC ""-//W3C//DTD SVG 1.1//EN"" ""http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd"" four stars This rating was published 02/28/201..."
1,quac,train,6442,5,What languages and cultures did he helped with?,What languages and cultures did Benjamin Lee Whorf help with?,1,54381058,54381058,54381058,http://web.archive.org/web/20191220074843id_/https://en.wikipedia.org/wiki/Benjamin_Lee_Whorf_p1,"Throughout his life Whorf was a chemical engineer by profession, but as a young man he took up an interest in linguistics. At first this interest drew him to the study of Bibli..."
2,trec,test,75,3,What was their importance in native cultures,What was the importance of turkey in native cultures?,1,54541736,54541736,54541736,http://web.archive.org/web/20200223002038id_/https://wanderingbull.com/turkey-feathers-history/_p2,"Roles of Turkey Feathers in Native American Culture Native Americans first domesticated turkeys around 800 B.C., Turkeys weren’t initially used for their meat, but rather their..."


The passage lookup resolves one gold passage from each QReCC source dataset. Matching `passage_id`, `row_index`, and `id` values confirm consistent identifier mapping between the processed splits and local corpus.

### Persisted ANCE encoding analysis

`load_ance_passage_encoding_analysis()` reads the previously generated JSON artifact without loading the corpus or rerunning tokenization. It reconstructs five pandas tables covering corpus size, token-length distribution, unknown tokens, truncation, and batch padding. The original run metadata is restored alongside these summaries.

The next cell displays the saved original corpus-encoding profile, not measurements of a new index build.

In [13]:
ANCE_ANALYSIS_OUT = project_path(
    "experiments",
    "results",
    "ance_passage_encoding",
    "qrecc.json",
)

ance_passage_analysis = rocc.load_ance_passage_encoding_analysis(ANCE_ANALYSIS_OUT)

display(ance_passage_analysis.corpus_summary)
display(ance_passage_analysis.token_length_summary)
display(ance_passage_analysis.unk_summary)
display(ance_passage_analysis.truncation_summary)
display(ance_passage_analysis.padding_summary)

,dataset_name,num_passages,expected_total_passages,max_passages,full_scan,batch_size,max_length_setting,workers,chunk_batches,encoder_name,source_file
0,qrecc,54573064,54573064,None,True,160,384,32,4,castorini/ance-msmarco-passage,/experiments/data/qrecc/collection/qrecc_collection.jsonl


,mean,variance,ci95_low,ci95_high,min,max
0,356.443345,1.297543e+06,356.141122,356.745568,2,1103254


,unk_tokens,visible_tokens,unk_rate,passages_with_unk,passages_with_unk_fraction
0,9,19452205496,4.626725e-10,7,1.282684e-07


,truncated_passages,truncation_rate,truncated_tokens_total,mean_truncated_tokens_all,mean_truncated_tokens_truncated_only
0,11095334,0.203312,2940311084,53.878431,265.004288


,batch_padded_length_mean,batch_padded_length_variance,batch_padded_length_ci95_low,batch_padded_length_ci95_high,batch_padded_length_min,batch_padded_length_max,actual_visible_tokens_total,padded_tokens_total,padding_overhead_tokens_total,padding_overhead_fraction,effective_token_utilization
0,383.95183,2.993279,383.946023,383.957636,255,384,16511894412,20953427776,4441533364,0.211972,0.788028


All **54,573,064 QReCC source passages** were analyzed with the ANCE tokenizer at a 384-token limit. Mean passage length is 356.44 tokens and unknown tokens are negligible. **20.33%** of passages are truncated. Batch padding contributes **21.20% overhead**, yielding **78.80% token utilization**.

### QReCC BM25 index

Exposes the local QReCC collection to Pyserini without duplicating the corpus, then verifies the Lucene index, document count, and build log. The thesis BM25 index contains **54,241,550 passages**, after excluding **331,474 empty passages** and **40 passages that become empty after text cleaning** from the source corpus. If the collection is unavailable, only the existing index is inspected.

In [14]:
QRECC_BM25_INDEX_DIR = project_path(
    "experiments",
    "data",
    "pyserini_bm25_lucene_qrecc",
    "lucene_index",
)
QRECC_BM25_LOG_FILE = project_path(
    "experiments",
    "logs",
    "qrecc_bm25_lucene_index.log",
)

if qrecc_collection.collection_jsonl.exists():
    qrecc_bm25_collection = rocc.write_qrecc_bm25_collection(
        qrecc_collection,
        progress=True,
    )
    qrecc_bm25_source_passages = qrecc_bm25_collection.written_passages
else:
    qrecc_bm25_collection = None
    qrecc_bm25_source_passages = None
    print("BM25 collection symlink skipped: qrecc_collection.jsonl is not local.")
qrecc_bm25_index_status = rocc.inspect_bm25_lucene_index(
    QRECC_BM25_INDEX_DIR,
    source_passages=qrecc_bm25_source_passages,
    log_file=QRECC_BM25_LOG_FILE,
)

if qrecc_bm25_collection is not None:
    display(pd.DataFrame([{
        "output_file": qrecc_bm25_collection.output_file,
        "written_passages": qrecc_bm25_collection.written_passages,
        "skipped_existing": qrecc_bm25_collection.skipped_existing,
        "is_symlink": qrecc_bm25_collection.output_file.is_symlink(),
        "source": qrecc_bm25_collection.output_file.resolve(),
        "bm25_index_complete": qrecc_bm25_index_status.complete,
        "bm25_index_docs": qrecc_bm25_index_status.num_docs,
    }]))
display(pd.DataFrame([qrecc_bm25_index_status.to_dict()]))


,output_file,written_passages,skipped_existing,is_symlink,source,bm25_index_complete,bm25_index_docs
0,/experiments/data/pyserini_bm25_lucene_qrecc/collection/docs.jsonl,54573064,True,True,/experiments/data/qrecc/collection/qrecc_collection.jsonl,True,54241550


,index_dir,exists,loadable,complete,num_docs,source_passages,log_file,log_exit_code,log_indexing_complete,log_indexed_docs,log_empty_docs,error
0,/experiments/data/pyserini_bm25_lucene_qrecc/lucene_index,True,True,True,54241550,54573064,/experiments/logs/qrecc_bm25_lucene_index.log,None,False,None,None,None


In [15]:
REBUILD_QRECC_BM25_INDEX = False

QRECC_BM25_INDEX_DIR = project_path(
    "experiments",
    "data",
    "pyserini_bm25_lucene_qrecc",
    "lucene_index",
)
QRECC_BM25_LOG_FILE = project_path(
    "experiments",
    "logs",
    "qrecc_bm25_lucene_index.log",
)

qrecc_bm25_index_status = rocc.inspect_bm25_lucene_index(
    QRECC_BM25_INDEX_DIR,
    source_passages=qrecc_bm25_source_passages,
    log_file=QRECC_BM25_LOG_FILE,
)

if qrecc_bm25_index_status.complete:
    print("BM25 Lucene index already complete:", QRECC_BM25_INDEX_DIR)
    qrecc_bm25_index = rocc.IndexBuildResult(
        index_dir=QRECC_BM25_INDEX_DIR,
        command=None,
        built=False,
        log_file=QRECC_BM25_LOG_FILE,
    )
elif REBUILD_QRECC_BM25_INDEX and qrecc_bm25_collection is not None:
    qrecc_bm25_index = rocc.build_bm25_lucene_index(
        collection_dir=qrecc_bm25_collection.output_file.parent,
        index_dir=QRECC_BM25_INDEX_DIR,
        threads=8,
        progress=True,
        log_file=QRECC_BM25_LOG_FILE,
        force=qrecc_bm25_index_status.exists,
    )
    qrecc_bm25_index_status = rocc.inspect_bm25_lucene_index(
        QRECC_BM25_INDEX_DIR,
        source_passages=qrecc_bm25_source_passages,
        log_file=QRECC_BM25_LOG_FILE,
    )
elif REBUILD_QRECC_BM25_INDEX:
    qrecc_bm25_index = None
    print("BM25 rebuild skipped: local qrecc_collection.jsonl is missing.")
else:
    qrecc_bm25_index = None
    print("BM25 Lucene index is missing or incomplete. Set REBUILD_QRECC_BM25_INDEX=True to rebuild explicitly.")

display(pd.DataFrame([qrecc_bm25_index_status.to_dict()]))
if qrecc_bm25_index is not None:
    display(pd.DataFrame([qrecc_bm25_index.__dict__]))


BM25 Lucene index already complete: <original-checkout>/experiments/data/pyserini_bm25_lucene_qrecc/lucene_index


,index_dir,exists,loadable,complete,num_docs,source_passages,log_file,log_exit_code,log_indexing_complete,log_indexed_docs,log_empty_docs,error
0,/experiments/data/pyserini_bm25_lucene_qrecc/lucene_index,True,True,True,54241550,54573064,/experiments/logs/qrecc_bm25_lucene_index.log,None,False,None,None,None


,index_dir,command,built,log_file
0,/experiments/data/pyserini_bm25_lucene_qrecc/lucene_index,None,False,/experiments/logs/qrecc_bm25_lucene_index.log


### BM25 retrieval check

A random gold-evaluable rewrite is searched against the verified Lucene index using \(k_1=0.82\), \(b=0.68\), and depth 100. Scores are min–max normalized and retrieved gold passages are marked.

In [16]:
QRECC_BM25_K1 = 0.82
QRECC_BM25_B = 0.68
QRECC_BM25_TOP_K = 100

qrecc_bm25_index_status = rocc.inspect_bm25_lucene_index(
    QRECC_BM25_INDEX_DIR,
    source_passages=globals().get("qrecc_bm25_source_passages"),
    log_file=QRECC_BM25_LOG_FILE,
)

if not qrecc_bm25_index_status.loadable:
    qrecc_bm25_retriever = None
    qrecc_bm25_hits_df = None
    print("BM25 inference skipped: local Lucene index is missing or not loadable.")
    display(pd.DataFrame([qrecc_bm25_index_status.to_dict()]))
else:
    qrecc_bm25_retriever = rocc.load_bm25_retriever(
        index_dir=QRECC_BM25_INDEX_DIR,
        k1=QRECC_BM25_K1,
        b=QRECC_BM25_B,
    )

    qrecc_query_row = qrecc_frame.sample(1).iloc[0]
    qrecc_query = qrecc_query_row["rewrite"]
    qrecc_gold_ids = {str(pid) for pid in qrecc_query_row["positive_ctx_passage_ids"]}

    qrecc_bm25_hits = qrecc_bm25_retriever.search(
        qrecc_query,
        top_k=QRECC_BM25_TOP_K,
        include_raw=True,
    )
    qrecc_bm25_hits = rocc.normalize_retrieval_scores(
        qrecc_bm25_hits,
        method="minmax",
    )
    qrecc_bm25_hits_df = pd.DataFrame(qrecc_bm25_hits)
    qrecc_bm25_hits_df["is_gold"] = qrecc_bm25_hits_df["docid"].astype(str).isin(qrecc_gold_ids)

    display(pd.DataFrame([{
        "split": qrecc_query_row["split"],
        "conversation_source": qrecc_query_row["conversation_source"],
        "conv_id": qrecc_query_row["conv_id"],
        "turn_id": qrecc_query_row["turn_id"],
        "question": qrecc_query_row["question"],
        "rewrite": qrecc_query,
        "gold_ids": sorted(qrecc_gold_ids),
    }]))
    display(qrecc_bm25_hits_df)



Aug 19, 2026 8:48:39 AM org.apache.lucene.internal.vectorization.PanamaVectorizationProvider <init>
INFO: Java vector incubator API enabled; uses preferredBitSize=256; FMA enabled


,split,conversation_source,conv_id,turn_id,question,rewrite,gold_ids
0,train,quac,2685,4,Were there any singles released from this album?,Were there any singles released from the album Safe as Milk by Captain Beefheart?,"[54295923, 54549463]"


,rank,docid,score,title,text,raw_passage_id,score_norm,is_gold
0,1,54295923,31.790100,None,"Safe as Milk - Wikipedia CentralNotice Safe as Milk From Wikipedia, the free encyclopedia Jump to navigation Jump to search 1967 studio album by Captain Beefheart and his Magic...",http://web.archive.org/web/20190619061711id_/https://en.wikipedia.org/wiki/Safe_as_Milk_p0,1.000000,True
1,2,54549463,30.219299,None,"Electricity (Captain Beefheart song) - Wikipedia CentralNotice Electricity (Captain Beefheart song) From Wikipedia, the free encyclopedia Jump to navigation Jump to search ""Ele...",http://web.archive.org/web/20201111225844id_/http://en.wikipedia.org/wiki/Electricity_(Captain_Beefheart_song)_p0,0.896906,True
2,3,46341094,28.336399,None,Safe As Milk – Advertising – Captain Beefheart Radar Station Skip to content Captain Beefheart Radar Station Captain Beefheart / Don Van Vliet fan site Primary Navigation Menu ...,http://www.beefheart.com/safe-as-milk-advertising/_p0,0.773329,False
3,4,54403043,26.826099,None,"Tom Waits ' shift in artistic direction, starting with 1983's Swordfishtrombones , was, Waits claims, a result of his wife Kathleen Brennan introducing him to Van Vliet's music...",http://web.archive.org/web/20191220122228id_/https://en.wikipedia.org/wiki/Captain_Beefheart_p40,0.674206,False
4,5,10917969,26.189199,None,"Wiggins, Little Roy Willet, Slim Wray, Link Yardbirds, The Yellow Balloon, The Young Rascals, The Youngbloods Genre Private Pressings & Outsider Art Blues Country/Rockabilly Ga...",https://sundazed.com/p/1319-Captain-Beefheart-and-His-Magic-Band-Plastic-Factory-/-Where-There-s-Woman-LIMITED-EDITION-BLACK-VINYL-7-Single.aspx_p5,0.632405,False
...,...,...,...,...,...,...,...,...
95,96,6344019,16.638000,None,My namesake and favorite band. I even have the officlal $150 Mr. Bungle clown mask. A perfect reimagining of quirky prog acts like zappa. - BungleBoy69 This is just one of Patt...,https://www.thetoptens.com/best-avant-garde-rock-bands/_p2,0.005546,False
96,97,29501658,16.613501,None,"The remainder of the album is definitely not bad, it's just very generic faux-blues and pop R&B music. And I have to say (and I hate to pick on one guy like this) that the saxo...",http://www.progarchives.com/review.asp?id=825682_p1,0.003938,False
97,98,38029845,16.613499,None,"The remainder of the album is definitely not bad, it's just very generic faux-blues and pop R&B music. And I have to say (and I hate to pick on one guy like this) that the saxo...",http://www.progarchives.com/Review.asp?id=825682_p1,0.003938,False
98,99,9730991,16.598400,None,3 days ago UnderTheCovers Various Fast 'N' Bulbous A Tribute To Captain Beefheart - *Various Fast 'N' Bulbous A Tribute To Captain Beefheart* *Get It At Discogs* *01 The Dog Fa...,http://oneman1001albums2.blogspot.com/2018/03/my-bloody-valentine-loveless.html?showComment=1521642096709_p4,0.002947,False


The original question is context-dependent because “this album” is unresolved. The standalone rewrite restores the missing entity as *Safe as Milk by Captain Beefheart*, allowing BM25 to retrieve both annotated gold passages at ranks **1 and 2**.

The raw BM25 scores are additionally min–max normalized to $[0,1]$ for inspection without changing the ranking. The `is_gold` column verifies document-ID alignment between the QReCC labels and Lucene index. Since the first relevant passage appears at rank 1, this query has a reciprocal rank of **1.0**.

### Inspect the local ANCE index

Place the completed public Drive folder contents directly in the displayed index directory. Incomplete uploads/copies are not ready for evaluation.

In [ ]:
QRECC_ANCE_LOCAL_INDEX_DIR = project_path(
    "experiments", "data", "pyserini_ance_faiss_qrecc", "faiss_flat_index_full_sharded")
qrecc_ance_shard_status = rocc.inspect_qrecc_ance_shards(QRECC_ANCE_LOCAL_INDEX_DIR)
display(pd.DataFrame([qrecc_ance_shard_status.to_dict()]))
print("Use a fully copied public index here, or explicitly start the local build below.")


### Build a local ANCE range

A build starts only when the local collection is available and `START_QRECC_ANCE_RANGE_BUILD=True`. Shards retain original IDs and forward corpus order and remain on local disk. Matching completed shards are reused. The explicit flag prevents an accidental full-corpus build.

In [ ]:
QRECC_ANCE_RANGE_INDEX_DIR = project_path(
    "experiments",
    "data",
    "pyserini_ance_faiss_qrecc",
    "faiss_flat_index_full_sharded",
)
QRECC_ANCE_DEVICE = "cuda:0" if __import__("torch").cuda.is_available() else "cpu"
QRECC_ANCE_BATCH_SIZE = 160
QRECC_ANCE_BATCHES_PER_SHARD = 625
QRECC_ANCE_PASSAGES_PER_SHARD = QRECC_ANCE_BATCH_SIZE * QRECC_ANCE_BATCHES_PER_SHARD

QRECC_ANCE_START_PASSAGE = 0
QRECC_ANCE_END_PASSAGE = qrecc_collection.num_passages
QRECC_ANCE_SHARD_ORDER = "forward"  # local resumable build uses forward order
QRECC_ANCE_OUTPUT_SHARD_OFFSET = QRECC_ANCE_START_PASSAGE // QRECC_ANCE_PASSAGES_PER_SHARD
QRECC_ANCE_DRY_RUN = False

qrecc_expected_ance_shards = (qrecc_collection.num_passages + 100_000 - 1) // 100_000
qrecc_local_ance_ready = (
    qrecc_ance_shard_status.complete_shards >= qrecc_expected_ance_shards
    and qrecc_ance_shard_status.indexed_passages >= qrecc_collection.num_passages
)
qrecc_ance_index_ready = bool(
    qrecc_local_ance_ready
)
qrecc_ance_colab_needed = not qrecc_ance_index_ready
qrecc_ance_colab_possible = qrecc_ance_colab_needed and qrecc_collection.collection_jsonl.exists()

START_QRECC_ANCE_RANGE_BUILD = False

display(pd.DataFrame([{
    "ance_index_ready": qrecc_ance_index_ready,
    "colab_build_needed": qrecc_ance_colab_needed,
    "local_collection_available": qrecc_collection.collection_jsonl.exists(),
    "colab_build_possible": qrecc_ance_colab_possible,
    "start_requested": START_QRECC_ANCE_RANGE_BUILD,
    "output_dir": QRECC_ANCE_RANGE_INDEX_DIR,
    "start_passage": QRECC_ANCE_START_PASSAGE,
    "end_passage": QRECC_ANCE_END_PASSAGE,
    "output_shard_offset": QRECC_ANCE_OUTPUT_SHARD_OFFSET,
    "shard_order": QRECC_ANCE_SHARD_ORDER,
    "passages_per_shard": QRECC_ANCE_PASSAGES_PER_SHARD,
    "dry_run": QRECC_ANCE_DRY_RUN,
}]))

if START_QRECC_ANCE_RANGE_BUILD and qrecc_ance_colab_possible:
    qrecc_ance_range_build = rocc.build_qrecc_ance_faiss_shards(
        collection=qrecc_collection,
        output_dir=QRECC_ANCE_RANGE_INDEX_DIR,
        start_passage=QRECC_ANCE_START_PASSAGE,
        end_passage=QRECC_ANCE_END_PASSAGE,
        output_shard_offset=QRECC_ANCE_OUTPUT_SHARD_OFFSET,
        device=QRECC_ANCE_DEVICE,
        batch_size=QRECC_ANCE_BATCH_SIZE,
        batches_per_shard=QRECC_ANCE_BATCHES_PER_SHARD,
        shard_order=QRECC_ANCE_SHARD_ORDER,
        progress=True,
        dry_run=QRECC_ANCE_DRY_RUN,
    )
    display(pd.DataFrame([qrecc_ance_range_build.__dict__]))
elif qrecc_ance_index_ready:
    print("QReCC ANCE build skipped: index is already complete.")
elif not qrecc_collection.collection_jsonl.exists():
    print("QReCC ANCE build skipped: local qrecc_collection.jsonl is missing.")
elif not START_QRECC_ANCE_RANGE_BUILD:
    print("QReCC ANCE range build is needed but not started. Set START_QRECC_ANCE_RANGE_BUILD=True explicitly.")
else:
    print("QReCC ANCE build skipped.")


### ANCE index verification

Inspects the local QReCC ANCE index and reports shard completeness, indexed passage count, and manifest status.

In [ ]:
qrecc_ance_shard_status = rocc.inspect_qrecc_ance_shards(QRECC_ANCE_LOCAL_INDEX_DIR)
display(pd.DataFrame([qrecc_ance_shard_status.to_dict()]))

,index_dir,complete_shards,first_shard_id,last_shard_id,next_shard_id,indexed_passages
0,/experiments/data/pyserini_ance_faiss_qrecc/faiss_flat_index_full_sharded,0,None,None,0,0
